In [ ]:
from pathlib import Path
from typing import Any, Callable

import geopandas as gpd
import h3
import pandas as pd
from shapely import Point, Polygon
from valhalla import Actor, get_config


In [2]:
def build_walk_query(
    location: dict[str, float], times: list[int] = (5, 10, 15)
) -> dict[str, Any]:
    return {
        "locations": [location],
        "costing": "pedestrian",
        "contours": [{"time": i} for i in times],
    }


def get_isochrone(query: dict, actor: Actor):
    result = {}
    isochrones = actor.isochrone(query)
    for contour_ix, isochrone in enumerate(isochrones["features"]):
        geom = isochrone["geometry"]["coordinates"]
        time = isochrone["properties"]["contour"]
        result[time] = Polygon(geom)
    return result


def determine_isochrones(
    points: gpd.GeoDataFrame,
    actor: Actor,
    times: list[int] = (5, 10, 15),
    costing: list[tuple[str, Callable]] = (("walk", build_walk_query),),
) -> pd.DataFrame:
    isochrone = []
    for row in points.itertuples():
        for label, func in costing:
            r = get_isochrone(
                func({"lon": row.lon, "lat": row.lat}, times=times),
                actor,
            )
            for t in times:
                if t in r:
                    isochrone.append([row.h3, r[t], label, t])
                else:
                    isochrone.append([row.h3, None, label, t])
    return pd.DataFrame.from_records(
        isochrone, columns=["h3", "geometry", "costing", "range"]
    )


def h3_to_point(cell: str) -> Point:
    return Point(h3.cell_to_latlng(cell)[::-1])


def h3_to_latlon(cell: str) -> tuple[float, float]:
    return h3.cell_to_latlng(cell)

In [34]:
RES = 10
AREA = "budapest_region"

In [35]:
hexagons = gpd.read_file(f"../data/h3/{AREA}/{RES}.geojson")
hexagons.rename(columns={"h3_polyfill": "h3"}, inplace=True)
hexagons["lat"], hexagons["lon"] = zip(*hexagons["h3"].apply(h3_to_latlon))

In [37]:
config = get_config(
    tile_extract="../data/valhalla/hungary/valhalla_tiles.tar",
    tile_dir="../data/valhalla/hungary/",
)
actor = Actor(config)

2026-10-27 18:29:15.284566254 [INFO] Tile extract successfully loaded with tile count: 256
2026-10-27 18:29:15.284688001 [WARN] (stat): /data/valhalla/traffic.tar No such file or directory
2026-10-27 18:29:15.284695709 [WARN] Traffic tile extract could not be loaded


In [ ]:
Path(f"../output/{AREA}/isochrones/").mkdir(parents=True, exist_ok=True)

isochrone_w15 = determine_isochrones(
    hexagons,
    actor,
    times=[15],
    costing=[("walk", build_walk_query)],
)
isochrone_w15 = gpd.GeoDataFrame(isochrone_w15, crs=4326)

isochrone_w15.to_csv(f"../output/{AREA}/isochrones/walk_area_hex{RES}.csv", index=False)
isochrone_w15.to_file(
    f"../output/{AREA}/isochrones/walk_area_hex{RES}.geojson", index=False
)

# for budapest
# runs 4s for level 8
# runs 26s for level 9
# runs 3m for level 10
# for budapest region
# runs 1m 7s for level 8
# runs 8m for level 9
# runs 56m for level 10